In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-2B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"],
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
vlm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

In [ ]:
SYSTEM = "Extract the student's registration number from the ID card. Respond with JSON only."

data = {
    "input": [
        """# ANNA UNIVERSITY
## Student Identity Card
**Name:** Priya Raman
**Department:** B.E. Computer Science
**Register No.:** 312221104087
**Batch:** 2021–2025
**Blood Group:** O+
**Emergency Contact:** 9840012345""",

        """| Field | Value |
|---|---|
| Student Name | Arjun Mehta |
| Programme | B.Tech Mechanical |
| Reg. ID | RA2111003010245 |
| Library Card | LIB-55821 |
| Valid Till | 05/2025 |""",

        """### SRM Institute of Science and Technology
Name: Kavya S
Course: MBA
Enrollment Number: 21MBA0193
Admission No: ADM/2021/7741
Hostel: Block C, Room 214""",
    ],
    "output": [
        '{"registration_number": "312221104087"}',
        '{"registration_number": "RA2111003010245"}',
        '{"registration_number": "21MBA0193"}',
    ],
}

TEST_CARD = """## VIT University — Student ID
**Name:** Rohan K
**Branch:** ECE
**Reg No:** 22BEC1043
**Mobile:** 9876501234
**Bus Route:** 14B"""

In [ ]:
tok = AutoTokenizer.from_pretrained(model_name)

def extract(text, max_new_tokens=64):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("BEFORE:", extract(TEST_CARD))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


BEFORE: {"registration_number": "22BEC1043"}


In [ ]:
from datasets import Dataset

MAX_LEN = 512

def build(ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["input"]}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(ex["output"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]   # loss only on the answer
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

ds = Dataset.from_dict(data).map(build, remove_columns=["input", "output"])

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (m - len(seq))
    return {
        "input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
        "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
    }

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear",   # covers Qwen3.5's attention + DeltaNet projections
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=10,            # deliberately overfit 3 samples to prove the loop works
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    logging_steps=1,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none",
)

Trainer(model=model, args=args, train_dataset=ds, data_collator=collate).train()

trainable params: 16,819,200 || all params: 1,898,644,288 || trainable%: 0.8859


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,0.142644
2,0.128365
3,0.086079
4,0.086079
5,0.142644
6,0.128365
7,0.086079
8,0.128365
9,0.004462
10,0.002296


TrainOutput(global_step=30, training_loss=0.031186842841642224, metrics={'train_runtime': 32.922, 'train_samples_per_second': 0.911, 'train_steps_per_second': 0.911, 'total_flos': 31193505784320.0, 'train_loss': 0.031186842841642224, 'epoch': 10.0})

In [ ]:
model.eval()
model.config.use_cache = True
print("AFTER:", extract(TEST_CARD))

model.save_pretrained("qwen35-2b-idcard-lora")   # adapter only, ~tens of MB
tok.save_pretrained("qwen35-2b-idcard-lora")

AFTER: {"registration_number": "22BEC1043"}


('qwen35-2b-idcard-lora/tokenizer_config.json',
 'qwen35-2b-idcard-lora/chat_template.jinja',
 'qwen35-2b-idcard-lora/tokenizer.json')